In [2]:
import duckdb
import pandas as pd
from lifetimes.utils import summary_data_from_transaction_data

con = duckdb.connect('../../warehouse.duckdb')

orders = con.execute("""
    select
        user_id,
        order_date,
        revenue
    from main.fct_orders
""").fetchdf()

summary = summary_data_from_transaction_data(
    orders,
    customer_id_col='user_id',
    datetime_col='order_date',
    monetary_value_col='revenue',
    observation_period_end=orders['order_date'].max()
)

summary.head()

,frequency,recency,T,monetary_value
user_id,,,,
u_000001,4.0,379.0,681.0,110.785
u_000010,0.0,0.0,505.0,0.000
u_000015,4.0,266.0,438.0,79.120
u_000024,0.0,0.0,417.0,0.000
u_000027,5.0,172.0,670.0,67.376


In [3]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

# BG/NBD needs customers with frequency >= 0, fits on everyone
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

print(bgf)

# Gamma-Gamma needs repeat customers only (frequency > 0) to model spend
returning_customers = summary[summary['frequency'] > 0]

ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

print(ggf)

<lifetimes.BetaGeoFitter: fitted with 1650 subjects, a: 1.85, alpha: 19.63, b: 2.87, r: 0.30>
<lifetimes.GammaGammaFitter: fitted with 861 subjects, p: 12.10, q: 2.61, v: 11.51>


In [4]:
summary['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365, summary['frequency'], summary['recency'], summary['T']
)

# Gamma-Gamma predicts avg order value; only meaningful for repeat customers,
# but the ggf conditional method safely handles frequency=0 rows too
summary['predicted_avg_order_value'] = ggf.conditional_expected_average_profit(
    summary['frequency'], summary['monetary_value']
)

summary['predicted_ltv_12m'] = (
    summary['predicted_purchases_12m'] * summary['predicted_avg_order_value']
)

summary.sort_values('predicted_ltv_12m', ascending=False).head(10)

,frequency,recency,T,monetary_value,predicted_purchases_12m,predicted_avg_order_value,predicted_ltv_12m
user_id,,,,,,,
u_001648,6.0,327.0,327.0,103.070000,3.269354,102.707688,335.787803
u_007972,5.0,102.0,114.0,84.458000,3.943982,84.508060,333.298309
u_008856,1.0,21.0,29.0,207.500000,1.663596,193.264094,321.513333
u_008091,6.0,143.0,176.0,95.035000,3.155266,94.847190,299.268094
u_006245,10.0,308.0,319.0,56.552000,5.213391,56.944178,296.872254
u_005261,12.0,326.0,358.0,57.455833,5.018826,57.773444,289.954872
u_008891,1.0,6.0,23.0,211.900000,1.405547,197.146907,277.099259
u_005416,6.0,209.0,245.0,92.478333,2.972748,92.346048,274.521526
u_007322,1.0,12.0,21.0,171.130000,1.699002,161.169117,273.826671


In [5]:
summary['ltv_decile'] = pd.qcut(summary['predicted_ltv_12m'], 10, labels=False, duplicates='drop')

channel_map = con.execute("""
    select
        o.user_id,
        o.first_touch_channel
    from main.fct_orders o
    qualify row_number() over (partition by o.user_id order by o.order_date) = 1
""").fetchdf()

summary_with_channel = summary.reset_index().merge(channel_map, on='user_id', how='left')

decile_by_channel = (
    summary_with_channel
    .groupby(['ltv_decile', 'first_touch_channel'])
    .size()
    .reset_index(name='count')
)

top_decile_channel_mix = (
    summary_with_channel[summary_with_channel['ltv_decile'] == 9]
    .groupby('first_touch_channel')
    .agg(customers=('user_id', 'count'), avg_predicted_ltv=('predicted_ltv_12m', 'mean'))
    .sort_values('avg_predicted_ltv', ascending=False)
)

top_decile_channel_mix

,customers,avg_predicted_ltv
first_touch_channel,,
meta_instagram,1,202.085738
meta_facebook,27,166.299031
tiktok_ads,10,148.208369
display_network,13,147.364002
email,19,144.970666
direct,37,141.967530
google_search,30,135.841282
affiliate,11,132.709639
organic_search,13,129.156514


In [6]:
roas = con.execute("""
    select channel, roas_last_touch, cac_last_touch
    from main.fct_channel_performance
    order by roas_last_touch desc nulls last
""").fetchdf()

roas

,channel,roas_last_touch,cac_last_touch
0,tiktok_ads,3.93,18.56
1,meta_facebook,2.84,24.89
2,meta_instagram,2.45,27.01
3,display_network,2.21,33.29
4,affiliate,1.33,55.49
5,google_search,1.04,69.16
6,organic_search,NaN,NaN
7,direct,NaN,NaN
8,email,NaN,NaN


In [7]:
con.close()